[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shhommychon/KorNorm/blob/feature%2Fbetter-g2pK/.dev_phonology/fix_pecab_word_arrow.ipynb)



###### 환경설정

In [1]:
!pip install -qq pecab pyarrow pandas g2pK

## EDA(?)

### 문제

In [2]:
from g2pk import G2p
g2p = G2p()
print()

print(g2p("느닷없이 비가 쏟아지던 날, 나는 수성못역 앞에서 컷워크를 연습하던 캣우먼 차림의 그녀를 바라보았다."))
print("^^^^^^                    ^^^^^^      ^^^^^          ^^^^^\n")

print(g2p("샬럿아말리에항 근처 캇웨이크해변에서 버릇없이 떠드는 관광객들을 피해, 끝없이 이어진 바다를 조용히 바라봤다."))
print("^^^^^^^^^^^     ^^^^^^^^^^^^ ^^^^^^                    ^^^^^\n")

print(g2p("친구는 갑자기 버빗원숭이 다큐멘터리 이야기를 꺼냈고, 겟앰프드를 하고 있던 나는 조금 밥맛없이 느껴졌다."))
print("           ^^^^^^^^                       ^^^^^^                   ^^^^^^\n")

print(g2p("한빛은행 옆 카페에서 우리는 컷인 타이밍과 풋워크 연습 이야기를 했고, 이건 춤 연습상 꽤 값있는 대화였다."))
print("^^^^^^                ^^^        ^^^^^                                ^^^^^\n")

print(g2p("달빛요정이라는 별명의 친구는 팟인코더로 편집한 영상을 보여주며 \"절치부심\"이라는 자작곡이라고 했다."))
print("^^^^^^^^^^^            ^^^^^^^^\n")

print(g2p("윗워터즈랜드에서 나온 음식은 기대보다 맛있었지만, 바로 옆 가게의 메뉴는 이상하게도 조금 맛없게 느껴졌다."))
print("^^^^^^^^^^^^                                                          ^^^^^\n")

print(g2p("이 아이디어를 이해한 사람이 몇이나 되는지 묻던 강사는, 성공하면 값없이 몇억만 벌겠다고 농담했다."))
print("                                                   ^^^^^ ^^^^^\n")

#   위 예제들은 MeCab 등 형태소 분석기 사전에 합성어가 단일 어휘로 통등재 되어 있을 때 발생하는
# g2pK의 음운 변동 오류 사례를 보여줌. 핵심 문제는 **표준 발음법 제15항**으로, 분석기가 합성어를
# 한 덩어리로 인식하면 형태소 경계가 소실되어, 대표음 교체 후 연음되는 대신 단순 연음(제13항)으로
# 처리되는 오류(예: '느닷없이' → `[느다섭씨]`)가 발생함.
#
#   런타임에서 분석기의 사전 로직을 동적으로 수정하는 것은 내 능지 상으로는 불가능에 가까우므로,
# 사전 빌드 단계에서 해당 방해 토큰들을 `KorNorm` 파이프라인 의존성인 `pecab`에서 제거하는
# 사전 패치 공정이 불가피함.

[nltk_data] Downloading package cmudict to /root/nltk_data...
[nltk_data]   Unzipping corpora/cmudict.zip.



느다섭씨 비가 쏘다지던 날, 나는 수성모셔 가페서 커숴크르 련스파던 캐수먼 차리믜 그녀를 바라보앋따.
^^^^^^                    ^^^^^^      ^^^^^          ^^^^^

샬러사말리에항 근처 카쉐이크해벼네서 버르섭씨 떠드는 관광객뜨를 피해, 끄섭씨 이어진 바다를 조용히 바라봗따.
^^^^^^^^^^^     ^^^^^^^^^^^^ ^^^^^^                    ^^^^^

친구는 갑짜기 버비숸숭이 다큐멘터리 이야기를 꺼낻꼬, 게샘프드를 하고 읻떤 나는 조금 밤마섭씨 느껴젇따.
           ^^^^^^^^                       ^^^^^^                   ^^^^^^

한비츤행 엽 카페에서 우리는 커신 타이밍과 푸숴크 연스 비야기를 핻꼬, 이건 추 면습쌍 꽤 갑씬는 대화엳따.
^^^^^^                ^^^        ^^^^^                                ^^^^^

달비쵸정이라는 별명의 친구는 파신코더로 편지파 녕상을 보여주며 "절치부심"이라는 자작꼬기라고 핻따.
^^^^^^^^^^^            ^^^^^^^^

위숴터즈랜드에서 나오 늠시근 기대보다 마시썯찌만, 바로 엽 까게의 메뉴느 니상하게도 조금 마섭께 느껴젇따.
^^^^^^^^^^^^                                                          ^^^^^

이 아이디어르 리해한 사라미 며치나 되는지 묻떤 강사는, 성공하면 갑썹씨 며청만 벌겓따고 농담핻따.
                                                   ^^^^^ ^^^^^



### `pecab` 사전 탐색

In [3]:
import os
import pandas as pd
import pyarrow as pa
import pecab

# # Pandas 출력 제한 해제 (모든 컬럼 확인용)
# pd.set_option("display.max_columns", None)
# pd.set_option("display.width", 1000)

# pecab 내부 리소스 경로 탐색
pecab_resource_dir = os.path.join(os.path.dirname(pecab.__file__), "_resources")
words_path = os.path.join(pecab_resource_dir, "words.arrow")

# Arrow 바이너리를 읽어 Pandas DataFrame으로 변환
words_table = pa.ipc.RecordBatchFileReader(pa.memory_map(words_path, 'r')).read_all()
df = words_table.to_pandas()

# df.to_csv("pecab.csv", encoding="utf-8-sig", index=False) # MacOS 엑셀로 보려면 "utf-8-sig" 필요
df.sample(10)

,surface,left_id,right_id,word_cost,POS,POS_type,morphemes
385934,사탕무,1780,3533,2639,NNG,COMP,"[('NNG', '사탕'), ('NNG', '무')]"
23104,園行,1780,3534,2639,NNG,MORP,None
654144,최장규,1788,3549,4104,NNP,MORP,None
721207,플레먼스,1788,3549,4104,NNP,MORP,None
214398,뉴이,1786,3545,2953,NNP,MORP,None
165740,귀여워,1804|1804,3|5,1711|2284,VA+EC|VA+EF,INFL|INFL,"[('VA', '귀엽'), ('EC', '어')]|[('VA', '귀엽'), ('E..."
271802,루벤알베스,1788,3549,4104,NNP,PREANY,"[('NNP', '루벤'), ('NNP', '알베스')]"
613684,조하만,1788,3550,4104,NNP,MORP,None
92951,製纖,1780,3534,2639,NNG,MORP,None
94098,解送,1780,3534,3069,NNG,MORP,None


In [4]:
import re
from jamo import hangul_to_jamo

def decompose_hangul(korean_string):
    return ''.join(hangul_to_jamo(korean_string))

TARGET = re.compile("(ᆩᄋ|ᆹᄋ|ᆺᄋ|ᆽᄋ|ᆾᄋ|ᆿᄋ|ᇀᄋ|ᇁᄋ|ᇂᄋ)")
def check_target(string):
    if TARGET.search(decompose_hangul(string)):
        return True
    return False

df_target = df[df.surface.apply(check_target)]
print(f"예외 대상 후보 건수: {len(df_target)}\n")
df_target.head(10)

예외 대상 후보 건수: 3245



,surface,left_id,right_id,word_cost,POS,POS_type,morphemes
120,ㄹ값에,2,3,2124,EC,MORP,None
215,ㄹ밖에,3,5,3990,EF,MORP,None
117758,가넷웨스턴,1788,3550,4104,NNP,PREANY,"[('NNP', '가넷'), ('NNP', '웨스턴')]"
118371,가량없이,735,2649,3337,MAG,MORP,None
118423,가렛에드워즈,1788,3549,4104,NNP,PREANY,"[('NNP', '가렛'), ('NNP', '에드워즈')]"
118424,가렛에반스,1788,3549,4104,NNP,PREANY,"[('NNP', '가렛'), ('NNP', '에반스')]"
118425,가렛에킨스,1788,3549,4104,NNP,PREANY,"[('NNP', '가렛'), ('NNP', '에킨스')]"
118426,가렛올슨,1788,3550,4104,NNP,PREANY,"[('NNP', '가렛'), ('NNP', '올슨')]"
118427,가렛웨버게일,1788,3550,4104,NNP,PREANY,"[('NNP', '가렛'), ('NNP', '웨버게일')]"
118428,가렛위건,1788,3550,4104,NNP,PREANY,"[('NNP', '가렛'), ('NNP', '위건')]"


In [5]:
import json
import os
from google import genai
from google.genai import types

def evaluate_g2p_chunks(chunked_data):
    # API 키 입력
    client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

    model = "gemini-2.5-flash"
    # Flash: "gemini-2.0-flash", "gemini-2.5-flash", "gemini-3-flash-preview"
    # Flash-Lite: "gemini-2.0-flash-lite", "gemini-2.5-flash-lite", "gemini-3.1-flash-lite-preview"
    # Pro: "gemini-2.5-pro", "gemini-3-pro-preview", "gemini-3.1-pro-preview"

    # LLM에게 부여할 시스템 프롬프트
    system_instruction = """
    당신은 한국어 음운론 및 표준 발음법 전문가입니다.
    주어진 단어와 G2P(Grapheme-to-Phoneme) 결과 쌍을 보고, 해당 G2P가 한국어 표준 발음법에 맞게 생성되었는지 객관적으로 평가하세요.

    [평가 기준 및 주의사항]
    1. 제13항 (연음 법칙): 앞말의 받침 뒤에 모음으로 시작되는 형식 형태소(조사, 어미, 접미사)가 올 경우 제 음가대로 연음한다.
       - 예: '가슴깊이' -> [가슴기피] (O), '몇이' -> [며치] (O), '가량없이' -> [가량업씨] (O)
    2. 제15항 (절음 법칙): 앞말의 받침 뒤에 모음 '아,어,오,우,위' 등으로 시작되는 실질 형태소가 올 경우 대표음으로 바꾼 뒤 연음한다.
       - 예: '덧없다' -> [더덥따] (O). 만약 G2P 결과가 [더섭따]라면 오답.
    3. 외래어 및 고유명사 복합어 경계: 각각 독립된 단어의 결합일 경우 경계에서 절음 법칙(대표음화)이 적용되어야 한다.
       - 예: '가넷웨스턴' -> [가넫] + [웨스턴] -> [가네뒈스턴] (O). 만약 G2P 결과가 [가네쉐스턴]이라면 오답.
    4. 제29항 ('ㄴ' 첨가): 합성어에서 뒷말이 '이,야,여,요,유'로 시작하면 'ㄴ' 소리가 첨가된다.
       - 예: '낯익다' -> [난닉따] (O). 만약 G2P 결과가 [나칙따]라면 오답.
    5. 예외 및 특수 사례:
       - '맛있다', '멋있다'는 [마디따/머디따] 외에 [마시따/머시따]도 표준 발음으로 허용.
       - '값어치'는 예외적으로 [가버치]로 발음. [갑서치]는 오답.
       - '며칠'은 항상 [며칠]로 발음.
    """

    user_prompt = f"다음 단어와 G2P 결과의 발음 정확성을 평가해 주세요:\n{json.dumps(chunked_data, ensure_ascii=False, indent=2)}"

    # JSON 출력을 강제하기 위한 스키마 정의
    response_schema = types.Schema(
        type=types.Type.ARRAY,
        items=types.Schema(
            type=types.Type.OBJECT,
            properties={
                "word": types.Schema(type=types.Type.STRING, description="평가한 단어"),
                "is_correct": types.Schema(type=types.Type.BOOLEAN, description="G2P 결과가 올바르면 true, 오류가 있으면 false"),
                "expected_pronunciation": types.Schema(type=types.Type.STRING, description="올바른 표준 발음"),
                "reason": types.Schema(type=types.Type.STRING, description="평가 사유 (줄바꿈 없이 한 줄로 짧게)")
            },
            required=["word", "is_correct", "expected_pronunciation", "reason"]
        )
    )

    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0.0, # 완벽한 객관성을 위해 온도를 0으로 설정
        response_mime_type="application/json",
        response_schema=response_schema,
        # thinking_config=types.ThinkingConfig(thinking_budget=1024) # 돈이 많다면, Pro 모델을 사용해 복잡한 음운 변동 추론을 위한 Thinking 활성화 가능
    )

    response = client.models.generate_content(
        model=model,
        contents=user_prompt,
        config=config,
    )

    return json.loads(response.text)

In [6]:
import time
import pandas as pd
from tqdm import tqdm

# 설정값
CHUNK_SIZE = 20
MAX_RETRIES = 5     # API 오류 시 최대 재시도 횟수
SLEEP_TIME = 10     # 무료 티어 RPM(Requests Per Minute) 제한을 피하기 위한 대기 시간 (초)

# 1. 평가할 단어 목록 추출
words_to_process = df_target["surface"].tolist()
all_evaluations = {}

print(f"총 {len(words_to_process)}개 단어 평가 시작 (청크 크기: {CHUNK_SIZE})...")

# 2. 20개 단위로 청크를 나누어 순회
for i in tqdm(range(0, len(words_to_process), CHUNK_SIZE), desc="G2P 평가 진행률"):
    chunk_words = words_to_process[i:i + CHUNK_SIZE]

    # API 요청을 위한 데이터 조립
    chunked_data = []
    for word in chunk_words:
        chunked_data.append({
            "word": word,
            "g2p_result": g2p(word)
        })

    # 3. API 호출 및 예외 처리 (재시도 로직)
    for attempt in range(MAX_RETRIES):
        try:
            results = evaluate_g2p_chunks(chunked_data)

            # 응답받은 결과를 딕셔너리에 매핑
            for res in results:
                parsed_word = res["word"]
                all_evaluations[parsed_word] = {
                    "is_correct": res.get("is_correct", False), # 기본값 False
                    "expected_pronunciation": res.get("expected_pronunciation", ""),
                    "reason": str(res.get("reason", "")).replace("\n", " ").strip()
                }

            break # 성공 시 재시도 루프 탈출

        except Exception as e:
            print(f"\n[오류 발생] 인덱스 {i}~{i+CHUNK_SIZE} (시도 {attempt + 1}/{MAX_RETRIES}): {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(10) # 오류 발생 시 서버가 진정할 수 있도록 10초 대기
            else:
                print("최대 재시도 횟수 초과. 해당 청크는 안전을 위해 오류 처리합니다.")
                for word in chunk_words:
                    all_evaluations[word] = {
                        "is_correct": False,
                        "expected_pronunciation": "API_ERROR",
                        "reason": "API_ERROR"
                    }

    # 무료 티어 API 호출 제한을 피하기 위해 매 청크마다 잠깐 대기
    time.sleep(SLEEP_TIME)

# 평가 결과를 원본 DataFrame에 새로운 컬럼으로 추가
# is_correct 값을 뒤집어서 is_problematic (삭제 대상 여부) 컬럼 생성
df_target["is_problematic"] = df_target["surface"].map(
    lambda x: not all_evaluations.get(x, {"is_correct": False})["is_correct"]
)
df_target["expected_pronunciation"] = df_target["surface"].map(
    lambda x: all_evaluations.get(x, {}).get("expected_pronunciation", "N/A")
)
df_target["reason"] = df_target["surface"].map(
    lambda x: all_evaluations.get(x, {}).get("reason", "API 응답 누락")
)

print("\n=== 평가 완료 ===")

# 최종 삭제 대상 필터링
df_suspect = df_target[df_target["is_problematic"] == True]

print(f"초기 의심 토큰 수: {len(df_target)}개")
print(f"최종 확정된 삭제 대상(is_problematic=True) 토큰 수: {len(df_suspect)}개")

# 드라이브에 안전하게 백업 저장
from google.colab import drive
drive.mount('/content/drive') # 구글 드라이브 마운트 (이미 되어있다면 생략 가능)

backup_path = "/content/drive/MyDrive/__df_target_backup.csv"
df_target.to_csv(backup_path, index=False, encoding="utf-8-sig") # MacOS 엑셀로 보려면 "utf-8-sig" 필요
print(f"결과가 성공적으로 저장되었습니다: {backup_path}")

# 결과 확인
display(df_target.head(10))

총 3245개 단어 평가 시작 (청크 크기: 20)...


G2P 평가 진행률: 100%|██████████| 163/163 [2:05:39<00:00, 46.26s/it]


=== 평가 완료 ===
초기 의심 토큰 수: 3245개
최종 확정된 삭제 대상(is_problematic=True) 토큰 수: 944개
결과가 성공적으로 저장되었습니다: /content/drive/MyDrive/__df_target_backup.csv



/tmp/ipykernel_262/953724092.py:123: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_target["is_problematic"] = df_target["surface"].map(
/tmp/ipykernel_262/953724092.py:126: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_target["expected_pronunciation"] = df_target["surface"].map(
/tmp/ipykernel_262/953724092.py:129: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: h

,surface,left_id,right_id,word_cost,POS,POS_type,morphemes,is_problematic,expected_pronunciation,reason
120,ㄹ값에,2,3,2124,EC,MORP,None,False,ㄹ갑쎄,겹받침 'ㅄ' 뒤에 모음으로 시작하는 조사 '에'가 오므로 'ㅅ'을 된소리로 연음함.
215,ㄹ밖에,3,5,3990,EF,MORP,None,False,ㄹ바께,받침 'ㄲ' 뒤에 모음으로 시작하는 조사 '에'가 오므로 제 음가대로 연음함.
117758,가넷웨스턴,1788,3550,4104,NNP,PREANY,"[('NNP', '가넷'), ('NNP', '웨스턴')]",True,가네뒈스턴,외래어 복합어 경계에서 앞말 받침 'ㅅ'은 대표음 [ㄷ]으로 바뀐 후 연음되어야 함.
118371,가량없이,735,2649,3337,MAG,MORP,None,False,가량업씨,'없'의 겹받침 'ㅄ' 뒤에 모음으로 시작하는 형식 형태소 '이'가 와서 [업씨]로...
118423,가렛에드워즈,1788,3549,4104,NNP,PREANY,"[('NNP', '가렛'), ('NNP', '에드워즈')]",True,가레데드워즈,고유명사 결합 시 앞말 받침 'ㅅ'은 대표음 [ㄷ]으로 변환 후 연음되어야 함.
118424,가렛에반스,1788,3549,4104,NNP,PREANY,"[('NNP', '가렛'), ('NNP', '에반스')]",True,가레데반스,고유명사 결합 시 앞말 받침 'ㅅ'은 대표음 [ㄷ]으로 변환 후 연음되어야 함.
118425,가렛에킨스,1788,3549,4104,NNP,PREANY,"[('NNP', '가렛'), ('NNP', '에킨스')]",True,가레데킨스,고유명사 결합 시 앞말 받침 'ㅅ'은 대표음 [ㄷ]으로 변환 후 연음되어야 함.
118426,가렛올슨,1788,3550,4104,NNP,PREANY,"[('NNP', '가렛'), ('NNP', '올슨')]",True,가레돌슨,고유명사 결합 시 앞말 받침 'ㅅ'은 대표음 [ㄷ]으로 변환 후 연음되어야 함.
118427,가렛웨버게일,1788,3550,4104,NNP,PREANY,"[('NNP', '가렛'), ('NNP', '웨버게일')]",True,가레뒈버게일,고유명사 결합 시 앞말 받침 'ㅅ'은 대표음 [ㄷ]으로 변환 후 연음되어야 함.
118428,가렛위건,1788,3550,4104,NNP,PREANY,"[('NNP', '가렛'), ('NNP', '위건')]",True,가레뒤건,고유명사 결합 시 앞말 받침 'ㅅ'은 대표음 [ㄷ]으로 변환 후 연음되어야 함.


In [7]:
for token in df_target[df_target["is_problematic"] == True]["surface"].tolist():
    print(f"    \"{token}\",")